# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
print("Available record sets in this dataset:")
for rs in dataset.record_sets:
    print(f"- Name: {rs.name} | @id: {rs.id}")

# As an example, print field @ids and names for each record set
print("\nFields in each record set:")
for rs in dataset.record_sets:
    print(f"\nRecord set: {rs.name} (@id: {rs.id})")
    for f in rs.fields:
        print(f"  - Field: {f.name} | @id: {f.id} | Data type: {f.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record set @ids dynamically from the dataset
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display loaded DataFrames and available columns for each
for record_set_id, df in dataframes.items():
    print(f"\nRecord Set @id: {record_set_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select the first numeric field in the first loaded record set
import numpy as np
if dataframes:
    # Select the first record set for analysis
    first_rs_id = next(iter(dataframes))
    df = dataframes[first_rs_id]
    # Find a numeric field (float or int type) by checking field dtypes
    numeric_field_id = None
    for col in df.columns:
        # Attempt to convert to numeric to check
        try:
            is_numeric = pd.api.types.is_numeric_dtype(df[col])
            if not is_numeric:
                # Try to coerce to numeric - if successful, treat as numeric
                pd.to_numeric(df[col].dropna().iloc[0:5])
                is_numeric = True
        except Exception:
            is_numeric = False
        if is_numeric:
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")

        # Convert column to numeric type if not already
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as an example
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized field '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another (non-numeric) column if available
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped filtered data by '{group_field}' and calculated mean {numeric_field_id}: ")
            display(grouped_df.head())
    else:
        print("No numeric field found in available data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 5))
    # Histogram of numeric field
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    # If group_field is present, boxplot
    if group_field:
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated use of the `mlcroissant` library to access and inspect the FAIR² dataset defined using a Croissant schema. 
- We loaded dataset metadata, discovered available record sets, fields, and their unique `@id` references, and explored the structure of the records in tabular format.
- A numeric field was selected for basic exploratory data analysis, including filtering, normalization, and grouping. Visualizations such as histograms and boxplots helped summarize the data distribution and group differences.
- You can extend this analysis by selecting other record sets, fields, or by building custom data processing pipelines for your own research or policy analyses based on this rich dataset.